In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [3]:
df=pd.read_csv('covid_toy.csv')

In [4]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [6]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [7]:
from sklearn.model_selection import train_test_split

x=df.drop(columns=['has_covid'])
y=df['has_covid']
x_train,x_test,y_train,y_test=train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=0
)

In [12]:
df.shape

(100, 6)

In [11]:
x_train.shape

(80, 5)

In [10]:
x_test.shape

(20, 5)

In [13]:
y_train.shape

(80,)

In [14]:
y_test.shape

(20,)

# 1. Column Transform one by one

In [15]:
x_train.head()

,age,gender,fever,cough,city
43,22,Female,99.0,Mild,Bangalore
62,56,Female,104.0,Strong,Bangalore
3,31,Female,98.0,Mild,Kolkata
71,75,Female,104.0,Strong,Delhi
45,72,Male,99.0,Mild,Bangalore


In [16]:
## Adding simple imputer to fever col
si=SimpleImputer() ## Use to handle missing data in col
x_train_fever=si.fit_transform(x_train[['fever']])

## also test
x_test_fever=si.fit_transform(x_test[['fever']])

In [18]:
x_train_fever.shape

(80, 1)

In [20]:
## OrdinalEncoding of Cough
oe=OrdinalEncoder(categories=[['Mild','Strong']])
x_train_cough=oe.fit_transform(x_train[['cough']])

## Also transform test
x_test_cough=oe.fit_transform(x_test[['cough']])


In [21]:
x_train_cough.shape

(80, 1)

In [23]:
## One Hot Encoding of City and Gender
ohe=OneHotEncoder(drop='first',sparse_output=False)
x_train_gender_city=ohe.fit_transform(x_train[['gender','city']])

## Also test
x_test_gender_city=ohe.fit_transform(x_test[['gender','city']])

In [24]:
x_train_gender_city.shape

(80, 4)

In [27]:
## Extracting Age
x_train_age=x_train.drop(columns=['gender','fever','cough','city']).values

## also test age
x_test_age=x_test.drop(columns=['gender','fever','cough','city']).values

x_train_age.shape

(80, 1)

In [28]:
x_train_transformed=np.concatenate((x_train_age,x_train_fever,x_train_gender_city,x_train_cough),axis=1)

x_test_transformed=np.concatenate((x_test_age,x_test_fever,x_test_gender_city,x_test_cough),axis=1)


In [29]:
x_train_transformed.shape

(80, 7)

In [30]:
x_test_transformed.shape

(20, 7)

# 2. Column Transformation in one step

In [33]:
df.sample(5)

,age,gender,fever,cough,city,has_covid
33,26,Female,98.0,Mild,Kolkata,No
82,24,Male,98.0,Mild,Kolkata,Yes
58,23,Male,98.0,Strong,Mumbai,Yes
24,13,Female,100.0,Strong,Kolkata,No
38,49,Female,101.0,Mild,Delhi,Yes


In [31]:
from sklearn.compose import ColumnTransformer

In [37]:
transformer=ColumnTransformer(
    transformers=[
        ('tnf1',SimpleImputer(),['fever']),
        ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
        ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
    ],
    remainder='passthrough'
)

In [39]:
transformer.fit_transform(x_train).shape

(80, 7)

In [40]:
transformer.transform(x_test).shape

(20, 7)